# Exploratory Data Analysis — DEAM Music Dataset

This notebook explores the DEAM (Database for Emotional Analysis of Music) dataset used for the GNN–BERT Music Context Understanding project.

**Contents:**
1. Dataset Overview & Loading
2. Valence/Arousal Distribution Analysis
3. OpenSMILE Feature Statistics
4. Graph Structure Analysis
5. Generated Tag Distribution

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
import json
from collections import Counter

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully!')

## 1. Dataset Overview & Loading

In [ ]:
# Load static annotations
DATA_DIR = '../data/raw'
ANNOTATIONS_DIR = os.path.join(DATA_DIR, 'annotations')

# If not extracted, extract from zip
if not os.path.exists(ANNOTATIONS_DIR):
    print('Extracting annotations...')
    zip_path = os.path.join('..', '..', 'DEAM_Annotations.zip')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('Done!')

# Load song-level annotations
ann_1_2000 = pd.read_csv(
    os.path.join(ANNOTATIONS_DIR, 'annotations averaged per song', 'song_level',
                 'static_annotations_averaged_songs_1_2000.csv'),
    skipinitialspace=True
)
ann_2000_2058 = pd.read_csv(
    os.path.join(ANNOTATIONS_DIR, 'annotations averaged per song', 'song_level',
                 'static_annotations_averaged_songs_2000_2058.csv'),
    skipinitialspace=True
)

# Standardize columns
ann_1_2000.columns = [c.strip() for c in ann_1_2000.columns]
ann_2000_2058.columns = [c.strip() for c in ann_2000_2058.columns]

# Keep common columns
common_cols = ['song_id', 'valence_mean', 'valence_std', 'arousal_mean', 'arousal_std']
ann_1_2000 = ann_1_2000[common_cols]
ann_2000_2058 = ann_2000_2058[common_cols]

# Combine
annotations = pd.concat([ann_1_2000, ann_2000_2058], ignore_index=True)
print(f'Total songs with annotations: {len(annotations)}')
print(f'\nAnnotation statistics:')
annotations.describe()

## 2. Valence/Arousal Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Valence distribution
axes[0].hist(annotations['valence_mean'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Valence (Mean)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Valence Ratings')
axes[0].axvline(x=5.0, color='red', linestyle='--', label='Neutral (5.0)')
axes[0].legend()

# Arousal distribution
axes[1].hist(annotations['arousal_mean'], bins=30, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Arousal (Mean)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Arousal Ratings')
axes[1].axvline(x=5.0, color='blue', linestyle='--', label='Neutral (5.0)')
axes[1].legend()

# V/A scatter (Russell's Circumplex)
scatter = axes[2].scatter(annotations['valence_mean'], annotations['arousal_mean'],
                          c=annotations['valence_mean'], cmap='RdYlGn', alpha=0.6, s=20)
axes[2].set_xlabel('Valence')
axes[2].set_ylabel('Arousal')
axes[2].set_title("Russell's Circumplex Model")
axes[2].axhline(y=5.0, color='gray', linestyle='--', alpha=0.5)
axes[2].axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)

# Add quadrant labels
axes[2].text(7.5, 7.5, 'Happy/\nEnergetic', ha='center', fontsize=9, fontweight='bold')
axes[2].text(2.5, 7.5, 'Angry/\nTense', ha='center', fontsize=9, fontweight='bold')
axes[2].text(7.5, 2.5, 'Relaxed/\nPeaceful', ha='center', fontsize=9, fontweight='bold')
axes[2].text(2.5, 2.5, 'Sad/\nCalm', ha='center', fontsize=9, fontweight='bold')

plt.colorbar(scatter, ax=axes[2], label='Valence')
plt.tight_layout()
plt.savefig('../results/plots/valence_arousal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/plots/valence_arousal_distribution.png')

In [ ]:
# V/A 2D density plot
fig, ax = plt.subplots(figsize=(8, 7))
sns.kdeplot(data=annotations, x='valence_mean', y='arousal_mean', 
            cmap='YlOrRd', fill=True, levels=20, ax=ax)
ax.set_xlabel('Valence', fontsize=14)
ax.set_ylabel('Arousal', fontsize=14)
ax.set_title('Valence-Arousal Density Distribution', fontsize=16)
ax.axhline(y=5.0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../results/plots/va_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. OpenSMILE Feature Statistics

In [ ]:
# Load a sample of features
FEATURES_DIR = os.path.join(DATA_DIR, 'features')
if not os.path.exists(FEATURES_DIR):
    print('Extracting features...')
    zip_path = os.path.join('..', '..', 'features.zip')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('Done!')

# Load sample features
feature_files = sorted([f for f in os.listdir(FEATURES_DIR) if f.endswith('.csv')])[:10]
print(f'Total feature files: {len(os.listdir(FEATURES_DIR))}')
print(f'\nSample feature file: {feature_files[0]}')

sample_df = pd.read_csv(os.path.join(FEATURES_DIR, feature_files[0]), sep=';')
print(f'Feature dimensions: {sample_df.shape}')
print(f'Columns (first 20): {list(sample_df.columns[:20])}')
print(f'\nTime range: {sample_df["frameTime"].min():.1f}s to {sample_df["frameTime"].max():.1f}s')
print(f'Time step: {sample_df["frameTime"].diff().iloc[1]:.2f}s')

In [ ]:
# Feature statistics across multiple songs
all_features = []
song_lengths = []
for f in feature_files:
    df = pd.read_csv(os.path.join(FEATURES_DIR, f), sep=';')
    all_features.append(df.iloc[:, 1:].values)  # Skip frameTime
    song_lengths.append(len(df))

all_feat_concat = np.concatenate(all_features, axis=0)
print(f'Combined feature matrix shape: {all_feat_concat.shape}')
print(f'Song lengths (frames): {song_lengths}')
print(f'Average song length: {np.mean(song_lengths):.0f} frames ({np.mean(song_lengths)*0.5:.0f}s)')

# Feature correlation heatmap (subset)
fig, ax = plt.subplots(figsize=(12, 10))
feat_names = list(sample_df.columns[1:21])
corr_matrix = np.corrcoef(all_feat_concat[:, :20].T)
sns.heatmap(corr_matrix, xticklabels=feat_names, yticklabels=feat_names,
            cmap='coolwarm', center=0, annot=False, ax=ax)
ax.set_title('Feature Correlation Matrix (First 20 Features)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/plots/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Graph Structure Analysis

In [ ]:
# Build sample graphs and analyze
from src.graph_builder import MusicGraphBuilder
import yaml

try:
    with open('../config.yaml') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    config = {
        'graph': {
            'similarity_threshold': 0.7,
            'max_edges_per_node': 10,
            'node_feature_dim': 260
        }
    }

builder = MusicGraphBuilder(config)

# Build graphs for sample songs
graph_stats = {'num_nodes': [], 'num_edges': [], 'avg_degree': []}
sample_graphs = []

for f in feature_files[:10]:
    df = pd.read_csv(os.path.join(FEATURES_DIR, f), sep=';')
    features = df.iloc[:, 1:].values.astype(np.float32)
    
    # Group into 2s segments (4 frames of 0.5s)
    n_segments = len(features) // 4
    if n_segments < 2:
        continue
    segments = np.array([features[i*4:(i+1)*4].mean(axis=0) for i in range(n_segments)])
    
    graph = builder.build_segment_graph(segments)
    sample_graphs.append(graph)
    
    num_nodes = graph.num_nodes
    num_edges = graph.edge_index.shape[1]
    avg_degree = num_edges / num_nodes if num_nodes > 0 else 0
    
    graph_stats['num_nodes'].append(num_nodes)
    graph_stats['num_edges'].append(num_edges)
    graph_stats['avg_degree'].append(avg_degree)

print('Graph Statistics (sample of 10 songs):')
for key, vals in graph_stats.items():
    print(f'  {key}: mean={np.mean(vals):.1f}, min={np.min(vals)}, max={np.max(vals)}')

In [ ]:
# Visualize graph statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(len(graph_stats['num_nodes'])), graph_stats['num_nodes'], color='#2ecc71')
axes[0].set_xlabel('Song Index')
axes[0].set_ylabel('Number of Nodes')
axes[0].set_title('Nodes per Graph')

axes[1].bar(range(len(graph_stats['num_edges'])), graph_stats['num_edges'], color='#3498db')
axes[1].set_xlabel('Song Index')
axes[1].set_ylabel('Number of Edges')
axes[1].set_title('Edges per Graph')

axes[2].bar(range(len(graph_stats['avg_degree'])), graph_stats['avg_degree'], color='#e74c3c')
axes[2].set_xlabel('Song Index')
axes[2].set_ylabel('Average Degree')
axes[2].set_title('Average Node Degree')

plt.tight_layout()
plt.savefig('../results/plots/graph_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Generated Tag Distribution

In [ ]:
# Generate tags and analyze distribution
from src.tag_generator import TagGenerator

tag_gen = TagGenerator(config)
tag_data = tag_gen.generate_all()

# Count tag frequencies
tag_vocab = tag_gen.get_tag_vocabulary()
tag_counts = Counter()
for song_id, info in tag_data.items():
    for tag in info['mood_tags'] + info['genre_tags']:
        tag_counts[tag] += 1

# Sort by frequency
tags_sorted = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(12, 5))
tag_names = [t[0] for t in tags_sorted]
tag_freqs = [t[1] for t in tags_sorted]
colors = ['#3498db' if t in tag_gen.mood_tags else '#e74c3c' for t in tag_names]
bars = ax.bar(tag_names, tag_freqs, color=colors, edgecolor='white')
ax.set_xlabel('Tag', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Generated Tag Distribution (Blue=Mood, Red=Genre)', fontsize=14)
plt.xticks(rotation=45, ha='right')

for bar, freq in zip(bars, tag_freqs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(freq), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/plots/tag_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show sample generated descriptions
print('Sample Generated Text Descriptions:')
print('=' * 80)
for i, (song_id, info) in enumerate(list(tag_data.items())[:5]):
    print(f"\nSong {song_id}:")
    print(f"  Valence: {info['valence']:.2f}, Arousal: {info['arousal']:.2f}")
    print(f"  Mood Tags: {info['mood_tags']}")
    print(f"  Genre Tags: {info['genre_tags']}")
    print(f"  Description: {info['text_description']}")

In [ ]:
# Tag co-occurrence matrix
all_tags = tag_gen.get_tag_vocabulary()
cooccurrence = np.zeros((len(all_tags), len(all_tags)))

for song_id, info in tag_data.items():
    song_tags = info['mood_tags'] + info['genre_tags']
    for t1 in song_tags:
        for t2 in song_tags:
            i = all_tags.index(t1)
            j = all_tags.index(t2)
            cooccurrence[i][j] += 1

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cooccurrence, xticklabels=all_tags, yticklabels=all_tags,
            cmap='YlOrRd', annot=True, fmt='.0f', ax=ax)
ax.set_title('Tag Co-occurrence Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../results/plots/tag_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEDA Complete! All plots saved to results/plots/')